# Part 2 – Supervised Machine Learning
This notebook implements the complete workflow required by the assignment. It expects `cleaned_data.csv` to be present in the same folder.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, r2_score, confusion_matrix,
    classification_report, roc_curve, roc_auc_score,
    precision_score, recall_score, f1_score
)

df=pd.read_csv("cleaned_data.csv")

# -------- Target definitions --------
num_cols=df.select_dtypes(include="number").columns.tolist()
target="Sales" if "Sales" in df.columns else num_cols[-1]

y_reg=df[target]
X=df.drop(columns=[target])

y_clf=(y_reg>y_reg.median()).astype(int)

# -------- Encoding --------
cat_cols=X.select_dtypes(include=["object","category"]).columns
X=pd.get_dummies(X,columns=cat_cols,drop_first=True)

# -------- Train/Test --------
X_train,X_test,y_reg_train,y_reg_test=train_test_split(
    X,y_reg,test_size=0.2,random_state=42)

_,_,y_clf_train,y_clf_test=train_test_split(
    X,y_clf,test_size=0.2,random_state=42)

scaler=StandardScaler()
X_train_s=scaler.fit_transform(X_train)
X_test_s=scaler.transform(X_test)

# -------- Linear Regression --------
lr=LinearRegression()
lr.fit(X_train_s,y_reg_train)
pred=lr.predict(X_test_s)

print("Linear Regression")
print("MSE:",mean_squared_error(y_reg_test,pred))
print("R2 :",r2_score(y_reg_test,pred))

coef=pd.DataFrame({"Feature":X.columns,"Coefficient":lr.coef_})
coef["Abs"]=coef["Coefficient"].abs()
print(coef.sort_values("Abs",ascending=False).head(3))

# -------- Ridge --------
ridge=Ridge(alpha=1.0)
ridge.fit(X_train_s,y_reg_train)
rpred=ridge.predict(X_test_s)

comparison=pd.DataFrame({
    "Model":["Linear Regression","Ridge"],
    "MSE":[mean_squared_error(y_reg_test,pred),
           mean_squared_error(y_reg_test,rpred)],
    "R2":[r2_score(y_reg_test,pred),
          r2_score(y_reg_test,rpred)]
})
print(comparison)

# -------- Logistic --------
print("Class counts")
print(y_clf_train.value_counts())

log1=LogisticRegression(max_iter=1000,class_weight="balanced")
log1.fit(X_train_s,y_clf_train)

proba=log1.predict_proba(X_test_s)[:,1]
pred_cls=(proba>=0.5).astype(int)

print(confusion_matrix(y_clf_test,pred_cls))
print(classification_report(y_clf_test,pred_cls))

auc=roc_auc_score(y_clf_test,proba)
fpr,tpr,_=roc_curve(y_clf_test,proba)

plt.figure(figsize=(6,4))
plt.plot(fpr,tpr,label=f"AUC={auc:.3f}")
plt.plot([0,1],[0,1],"--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# Threshold experiment
rows=[]
for th in np.arange(0.3,0.71,0.1):
    p=(proba>=th).astype(int)
    rows.append([round(th,2),
                 precision_score(y_clf_test,p,zero_division=0),
                 recall_score(y_clf_test,p,zero_division=0),
                 f1_score(y_clf_test,p,zero_division=0)])
th_df=pd.DataFrame(rows,columns=["Threshold","Precision","Recall","F1"])
print(th_df)

# Regularized Logistic
log2=LogisticRegression(C=0.01,max_iter=1000,class_weight="balanced")
log2.fit(X_train_s,y_clf_train)
proba2=log2.predict_proba(X_test_s)[:,1]

# Bootstrap CI
diff=[]
rng=np.random.default_rng(42)
for _ in range(500):
    idx=rng.choice(len(y_clf_test),len(y_clf_test),replace=True)
    d=roc_auc_score(y_clf_test.iloc[idx],proba[idx]) - roc_auc_score(y_clf_test.iloc[idx],proba2[idx])
    diff.append(d)

print("Mean AUC diff:",np.mean(diff))
print("95% CI:",np.percentile(diff,[2.5,97.5]))
